The function `countFrom(n)` creates the following *infinite stream* of natural numbers:
$$ [k \in \mathbb{N} \mid n \leq k] = [n,\; n+1,\; n+2,\; \cdots]$$

Java has no *generators*.  Instead, we use *iterators*.  An object of type `Iterator<Integer>` has two methods:
  * `hasNext()` checks whether there are more elements, while
  * `next()` returns the next element.

The function `countFrom` returns an object of an *anonymous class* that implements the interface `Iterator<Integer>`.  As the stream is infinite, the method `hasNext` always returns `true`.  The variable `k` stores the number that is returned by the next call of `next`.

In [ ]:
import java.util.function.*;

In [ ]:
Iterator<Integer> countFrom(int n) {
    return new Iterator<>() {
        int k = n;

        public boolean hasNext() {
            return true;
        }

        public Integer next() {
            return k++;
        }
    };
}

In [ ]:
var natNums = countFrom(0);

In [ ]:
natNums.next()

In [ ]:
natNums.next()

The function `printFirst` prints the first `n` numbers produced by the iterator `g`.

In [ ]:
void printFirst(Iterator<Integer> g, int n) {
    for (int i = 0; i < n; ++i) {
        System.out.println(g.next());
    }
}

In [ ]:
printFirst(countFrom(7), 3)

The function `filterStream` takes two arguments:
  * `g` is an infinite iterator of numbers,
  * `p` is a predicate.
  
It returns a new iterator that returns only those values `x` from `g`
such that `p(x)` is `true`.
$$ \texttt{filterStream}(G, p) = [ x \in G \mid p(x) ] $$
A predicate is an object of type `Predicate<Integer>`.  Its method `test` checks whether a number satisfies the predicate.

In [ ]:
Iterator<Integer> filterStream(Iterator<Integer> g, Predicate<Integer> p) {
    return new Iterator<>() {
        public boolean hasNext() {
            return true;
        }

        public Integer next() {
            while (true) {
                int x = g.next();
                if (p.test(x)) {
                    return x;
                }
            }
        }
    };
}

Let us test `filterStream` by creating a stream of numbers divisible by $3$.

In [ ]:
var divBy3 = filterStream(countFrom(1), n -> n % 3 == 0);
printFirst(divBy3, 5)

The function `filterPrime` takes a stream `g` of numbers, yields the first element `p`, filters all subsequent multiples of `p`, and calls `filterPrime` recursively on the resulting stream.
$$ \texttt{filterPrime}( [p, ... R]) = 
  \bigl[p, ...\texttt{filterPrime}([ x \in R \mid x \,\texttt{%}\, p \not= 0])\bigr] $$
When the method `next` of the iterator returned by `filterPrime(g)` is called for the first time, it takes the first number `p` from `g` and returns it.  At the same time, it creates the iterator
```
rest = filterPrime(filterStream(g, x -> x % p != 0))
```
All subsequent calls of `next` are delegated to `rest`.  As `rest` is only created when it is needed, the recursion does not run forever.

In [ ]:
Iterator<Integer> filterPrime(Iterator<Integer> g) {
    return new Iterator<>() {
        Iterator<Integer> rest = null;

        public boolean hasNext() {
            return true;
        }

        public Integer next() {
            if (rest == null) {
                int p = g.next();  // get first number and return it below, since it is prime
                // remove all multiples of prime p
                rest = filterPrime(filterStream(g, x -> x % p != 0));
                return p;
            }
            return rest.next();
        }
    };
}

The function `primes` generates an infinite sequence of *prime numbers* using 
the [*Sieve of Eratosthenes*](https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes).

In [ ]:
Iterator<Integer> primes() {
    return filterPrime(countFrom(2));
}

In [ ]:
printFirst(primes(), 100)